In [1]:
import torch
from transformers import AutoTokenizer
from src.model import XMistralForCausalLM

In [14]:
device = torch.device("cuda")
llm_name_or_path = "Hannibal046/xrag-7b"
retriever_name_or_path = 'salesforce/sfr-embedding-mistral'

In [3]:
model = XMistralForCausalLM.from_pretrained(llm_name_or_path,torch_dtype = torch.bfloat16,low_cpu_mem_usage = True,).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(llm_name_or_path,add_eos_token=False,use_fast=False,padding_side='left')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
from datasets import load_dataset

In [7]:
ds = load_dataset("brimmann2/squad_qa1", split="train")

In [10]:
retrieval_embed_length = 0
retriever,retriever_tokenizer = None,None

In [12]:
from src.model import SFR

In [ ]:
retriever = SFR.from_pretrained(retriever_name_or_path,torch_dtype = torch.bfloat16)
retriever_tokenizer = AutoTokenizer.from_pretrained(args.retriever_name_or_path)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [16]:
retrieval_embed_length = retriever.get_embed_length()
retriever_hidden_size = retriever.get_embed_dim()
retriever.eval()
retriever = retriever.to(device)

In [18]:
from src.language_modeling.utils import get_retrieval_embeds

In [ ]:
@torch.no_grad()
def prepare_retrieval_embeds(backgrounds,retriever,tokenizer,batch_size = 16):
    backgrounds = [backgrounds[idx:idx+batch_size] for idx in range(0,len(backgrounds),batch_size)]
    device = retriever.device
    ret = []
    for background in backgrounds:
        tokenized_retrieval_text = tokenizer(
            background, 
            max_length=180,
            padding=True, truncation=True, return_tensors="pt")
        
        ## return a torch tensor of shape [batch_size,d_model]
        embeds = get_retrieval_embeds(
            model = retriever,
            input_ids = tokenized_retrieval_text['input_ids'].to(device),
            attention_mask = tokenized_retrieval_text['attention_mask'].to(device),
        ).cpu()

        embeds = [embeds[idx] for idx in range(embeds.shape[0])]
        ret.extend(embeds)
    return ret

In [ ]:
_retrieval_embeds = prepare_retrieval_embeds(
            list(ds["text"]),
            retriever,
            retriever_tokenizer,
        )